# AI Handwriting — Multi-Task CNN (Character Recognition + Writer ID)
**Step 0:** Runtime → Change runtime type → **T4 GPU** before running anything.

In [ ]:
# Cell 1: Confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Clone or pull repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest changes...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

os.chdir(f'{REPO_PATH}/Data')
print('Working dir:', os.getcwd())

In [ ]:
# Cell 4: Copy Writers_pngs from Drive
# Your Writers_pngs folder must be at: My Drive/Writers_pngs/
import shutil, os

src  = '/content/drive/MyDrive/Writers_pngs'
dest = f'{REPO_PATH}/Data/Writers_pngs'

if os.path.exists(dest) and len(os.listdir(dest)) > 2:
    print('Writers_pngs already present — skipping copy')
elif os.path.exists(src):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print('Writers_pngs copied successfully')
else:
    print('ERROR: Writers_pngs not found at', src)
    raise FileNotFoundError(src)

skip = {'Writers_Zip', 'output_preview'}
total = 0
for d in sorted(os.listdir(dest)):
    if d in skip: continue
    p = os.path.join(dest, d)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.png')])
        print(f'  {d}: {n} samples')
        total += n
print(f'Total: {total} samples')

In [ ]:
# Cell 5: Clear old checkpoints, then train from scratch
import shutil, os

ckpt_dir = f'{REPO_PATH}/checkpoints'
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print('Old checkpoints cleared')
os.makedirs(ckpt_dir, exist_ok=True)

os.chdir(f'{REPO_PATH}/Data')
!python train.py

In [ ]:
# Cell 6: Save best_model.pt to Drive + download
import shutil, os
from google.colab import files

ckpt_src  = f'{REPO_PATH}/checkpoints/best_model.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'best_model.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('Download started — save best_model.pt into your local checkpoints/ folder')

In [ ]:
# Cell 7: Run evaluation — confusion matrix + t-SNE
os.chdir(f'{REPO_PATH}/Data')
!pip install scikit-learn -q
!python evaluate.py

# Download outputs
from google.colab import files
eval_dir = f'{REPO_PATH}/eval_output'
for fname in os.listdir(eval_dir):
    fpath = os.path.join(eval_dir, fname)
    files.download(fpath)
    print(f'Downloading: {fname}')

In [ ]:
# Cell 8: Demo — run inference on one writer folder
# Change WRITER to any writer folder name present in Writers_pngs
WRITER = 'writer_Abdullah'
writer_folder = f'{REPO_PATH}/Data/Writers_pngs/{WRITER}'

os.chdir(f'{REPO_PATH}/Data')
!python demo.py --writer_folder "{writer_folder}"